In [7]:
import json
import os
import time
from pathlib import Path

import pandas as pd
import requests
from dotenv import load_dotenv

load_dotenv("../.env")
API_KEY = os.environ["LASTFM_API_KEY"]

In [8]:
streams = pd.read_parquet("../data/processed/streams_clean.parquet")
artist_plays = streams["artist"].value_counts()
print(len(artist_plays), "unique artists")
artist_plays.head(10)

3341 unique artists


artist
Skrillex          1340
Kanye West        1277
Flume             1191
Kendrick Lamar    1021
A$AP Rocky         994
ISOxo              958
Knock2             907
Subtronics         803
GRiZ               788
ODESZA             773
Name: count, dtype: int64

In [9]:
LASTFM_URL = "https://ws.audioscrobbler.com/2.0/"

def get_top_tags(artist_name):
    resp = requests.get(LASTFM_URL, params={
        "method": "artist.gettoptags",
        "artist": artist_name,
        "api_key": API_KEY,
        "format": "json",
        "autocorrect": 1,
    }, timeout=10)
    resp.raise_for_status()
    return resp.json()

In [11]:
get_top_tags("Skrillex")

{'toptags': {'tag': [{'count': 100,
    'name': 'dubstep',
    'url': 'https://www.last.fm/tag/dubstep'},
   {'count': 65,
    'name': 'electronic',
    'url': 'https://www.last.fm/tag/electronic'},
   {'count': 20,
    'name': 'breakcore',
    'url': 'https://www.last.fm/tag/breakcore'},
   {'count': 17, 'name': 'electro', 'url': 'https://www.last.fm/tag/electro'},
   {'count': 7,
    'name': 'electronica',
    'url': 'https://www.last.fm/tag/electronica'},
   {'count': 5, 'name': 'Brostep', 'url': 'https://www.last.fm/tag/Brostep'},
   {'count': 3,
    'name': 'electro house',
    'url': 'https://www.last.fm/tag/electro+house'},
   {'count': 1, 'name': 'House', 'url': 'https://www.last.fm/tag/House'},
   {'count': 1, 'name': 'dance', 'url': 'https://www.last.fm/tag/dance'},
   {'count': 1,
    'name': 'fidget house',
    'url': 'https://www.last.fm/tag/fidget+house'}],
  '@attr': {'artist': 'Skrillex'}}}

In [13]:
CACHE_PATH = Path("../data/cache/artist_tags.json")
CACHE_PATH.parent.mkdir(parents=True, exist_ok=True)

tag_cache = json.loads(CACHE_PATH.read_text()) if CACHE_PATH.exists() else {}
print(len(tag_cache), "artists already cached")

def save_cache():
    CACHE_PATH.write_text(json.dumps(tag_cache))

0 artists already cached


In [14]:
todo = [a for a in artist_plays.index if a not in tag_cache]
print(len(todo), "artists to fetch")

failed = []
for i, artist in enumerate(todo, 1):
    try:
        data = get_top_tags(artist)
        tags = data.get("toptags", {}).get("tag", [])
        if isinstance(tags, dict):          # single-tag artists come back as a bare dict
            tags = [tags]
        tag_cache[artist] = [{"name": t["name"], "count": t["count"]} for t in tags]
    except requests.RequestException as e:
        failed.append(artist)
        print("FAILED:", artist, "-", e)
    time.sleep(0.2)
    if i % 100 == 0:
        save_cache()
        print(f"{i}/{len(todo)} done")

save_cache()
print(f"finished: {len(tag_cache)} cached, {len(failed)} failed")

3341 artists to fetch
100/3341 done
200/3341 done
300/3341 done
400/3341 done
500/3341 done
600/3341 done
700/3341 done
800/3341 done
900/3341 done
1000/3341 done
1100/3341 done
1200/3341 done
1300/3341 done
1400/3341 done
1500/3341 done
1600/3341 done
1700/3341 done
1800/3341 done
1900/3341 done
2000/3341 done
2100/3341 done
2200/3341 done
2300/3341 done
2400/3341 done
2500/3341 done
2600/3341 done
2700/3341 done
2800/3341 done
2900/3341 done
3000/3341 done
3100/3341 done
3200/3341 done
3300/3341 done
finished: 3341 cached, 0 failed


In [15]:
empty = [a for a, t in tag_cache.items() if not t]
print(f"{len(empty)} artists with no tags ({len(empty)/len(tag_cache):.1%})")

247 artists with no tags (7.4%)


In [16]:
untagged_share = artist_plays[artist_plays.index.isin(empty)].sum() / artist_plays.sum()
print(f"untagged share of streams: {untagged_share:.1%}")

untagged share of streams: 1.7%
